# Statistics Lab Manual
## Descriptive Statistics using Python (Jupyter Notebook)

**Dataset used:** California Housing Dataset (REAL dataset, 20,640 rows, from `scikit-learn`)

**Level:** 3rd Semester — simple, beginner-friendly Python code

**Topics:**
1. Problem Statement
2. Load Dataset
3. Dependent and Independent Variables
4. Mean vs Median
5. Variance vs IQR
6. Skewness and Distribution Shape
7. Histogram, Boxplot, Density Plot
8. Pivot Tables

---


## 1. Problem Statement

We want to study **house prices in California**.

A real estate company wants to know:
- What is the average / typical house price?
- Does house price depend on income, number of rooms, house age, or location?
- Is the house price data evenly spread, or are there very expensive outlier houses?

To answer this, we will use a **real dataset** called the **California Housing Dataset**.
It has **20,640 rows**, one row for each neighbourhood (called a "block group") in
California, collected from the 1990 US Census.

**Dependent variable (what we want to study):** Median House Value
**Independent variables (factors that may affect it):** Income, House Age, Rooms,
Population, Location, etc.

## 2. Load Dataset

`scikit-learn` already provides this real dataset. We just need to import it and load it
into a pandas DataFrame. (Internet is required the first time — after that it is cached
on your computer.)

In [ ]:
# Step 1: Import the libraries we need
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.datasets import fetch_california_housing

print("Libraries imported successfully")

In [ ]:
# Step 2: Load the real California Housing dataset
housing = fetch_california_housing(as_frame=True)
df = housing.frame

# Rename the target column to something easier to read
df = df.rename(columns={"MedHouseVal": "MedianHouseValue"})

# The target values are in units of $100,000, so multiply to get actual dollars
df["MedianHouseValue"] = df["MedianHouseValue"] * 100000

print("Dataset loaded successfully")
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

In [ ]:
# Step 3: Look at the first few rows
df.head()

In [ ]:
# Step 4: Basic information about the dataset
df.info()

In [ ]:
# Step 5: Check for missing values
df.isnull().sum()

## 3. Dependent and Independent Variables

| Variable | Type | Meaning |
|---|---|---|
| MedInc | Independent | Median income in the area |
| HouseAge | Independent | Median age of houses |
| AveRooms | Independent | Average rooms per house |
| AveBedrms | Independent | Average bedrooms per house |
| Population | Independent | Population of the area |
| AveOccup | Independent | Average people per household |
| Latitude | Independent | Location (north-south) |
| Longitude | Independent | Location (east-west) |
| **MedianHouseValue** | **Dependent** | **The house price we want to study** |

The **dependent variable (Y)** is the one we are trying to explain: `MedianHouseValue`.
The **independent variables (X)** are the factors that might affect it.

In [ ]:
# Dependent variable
Y = df["MedianHouseValue"]

# Independent variables
X = df.drop("MedianHouseValue", axis=1)

print("Dependent variable   :", "MedianHouseValue")
print("Independent variables:", list(X.columns))

## 4. Mean vs Median Comparison

- **Mean** = sum of values / number of values (average). Gets affected by very large or
  very small values (outliers).
- **Median** = the middle value when data is sorted. Not affected by outliers.

Rule of thumb:
- Mean ≈ Median → data is symmetric.
- Mean > Median → data is right-skewed (some very high values pull the mean up).
- Mean < Median → data is left-skewed (some very low values pull the mean down).

In [ ]:
# Mean and Median of MedianHouseValue
mean_value = df["MedianHouseValue"].mean()
median_value = df["MedianHouseValue"].median()

print("Mean of MedianHouseValue  :", mean_value)
print("Median of MedianHouseValue:", median_value)

if mean_value > median_value:
    print("Interpretation: Mean > Median -> data is RIGHT-SKEWED")
elif mean_value < median_value:
    print("Interpretation: Mean < Median -> data is LEFT-SKEWED")
else:
    print("Interpretation: Mean = Median -> data is SYMMETRIC")

In [ ]:
# Let's also check Mean vs Median for Median Income
mean_income = df["MedInc"].mean()
median_income = df["MedInc"].median()

print("Mean of MedInc  :", mean_income)
print("Median of MedInc:", median_income)

## 5. Variance vs IQR (Interquartile Range)

- **Variance** and **Standard Deviation** measure how spread out the data is from the
  mean. They use every value, so outliers affect them a lot.
- **IQR = Q3 - Q1** measures the spread of the middle 50% of the data. It is not
  affected much by outliers.

A big difference between what standard deviation suggests and what IQR suggests usually
means the data has outliers.

In [ ]:
# Variance and Standard Deviation
variance_value = df["MedianHouseValue"].var()
std_value = df["MedianHouseValue"].std()

print("Variance of MedianHouseValue         :", variance_value)
print("Standard Deviation of MedianHouseValue:", std_value)

In [ ]:
# IQR calculation
Q1 = df["MedianHouseValue"].quantile(0.25)
Q3 = df["MedianHouseValue"].quantile(0.75)
IQR = Q3 - Q1

print("Q1 (25th percentile):", Q1)
print("Q3 (75th percentile):", Q3)
print("IQR (Q3 - Q1)       :", IQR)

In [ ]:
# Finding outliers using the 1.5 x IQR rule
lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

outliers = df[(df["MedianHouseValue"] < lower_limit) | (df["MedianHouseValue"] > upper_limit)]

print("Lower limit:", lower_limit)
print("Upper limit:", upper_limit)
print("Number of outliers found:", len(outliers))

## 6. Skewness and Distribution Shape

**Skewness** tells us if the data is symmetric or leans to one side.

| Skewness value | Meaning |
|---|---|
| close to 0 | Symmetric distribution |
| greater than 0 | Right-skewed (long tail on the right) |
| less than 0 | Left-skewed (long tail on the left) |

In [ ]:
# Skewness of MedianHouseValue
skewness_value = df["MedianHouseValue"].skew()
print("Skewness of MedianHouseValue:", skewness_value)

if skewness_value > 0:
    print("Interpretation: Right-skewed distribution (long tail towards high prices)")
elif skewness_value < 0:
    print("Interpretation: Left-skewed distribution (long tail towards low prices)")
else:
    print("Interpretation: Symmetric distribution")

In [ ]:
# Skewness of a few more columns
print("Skewness of MedInc     :", df["MedInc"].skew())
print("Skewness of Population :", df["Population"].skew())
print("Skewness of AveRooms   :", df["AveRooms"].skew())

## 7. Histogram, Boxplot, and Density Plot

- **Histogram** shows how frequently values occur, using bars.
- **Boxplot** shows the median, quartiles, and outliers.
- **Density plot** is a smooth curve version of the histogram.

In [ ]:
# Histogram of MedianHouseValue
plt.figure(figsize=(8, 5))
plt.hist(df["MedianHouseValue"], bins=50, color="skyblue", edgecolor="black")
plt.title("Histogram of Median House Value")
plt.xlabel("Median House Value ($)")
plt.ylabel("Frequency")
plt.show()

In [ ]:
# Boxplot of MedianHouseValue
plt.figure(figsize=(6, 5))
sns.boxplot(y=df["MedianHouseValue"], color="lightgreen")
plt.title("Boxplot of Median House Value")
plt.ylabel("Median House Value ($)")
plt.show()

In [ ]:
# Density plot of MedianHouseValue
plt.figure(figsize=(8, 5))
sns.kdeplot(df["MedianHouseValue"], fill=True, color="orange")
plt.title("Density Plot of Median House Value")
plt.xlabel("Median House Value ($)")
plt.show()

**Interpretation:** The histogram and density plot both show that most houses are
low to medium priced, with a long tail of expensive houses on the right side — this
matches the positive skewness value we calculated earlier. The boxplot shows several
points above the upper whisker, confirming the presence of outliers (very expensive
houses).

## 8. Pivot Tables

A pivot table helps us summarize data by groups. We will first create a simple category
column for income, then build a pivot table showing average house value for each income
group.

In [ ]:
# Step 1: Create income categories (Low, Medium, High, Very High)
df["IncomeGroup"] = pd.qcut(df["MedInc"], q=4, labels=["Low", "Medium", "High", "Very High"])

df[["MedInc", "IncomeGroup"]].head()

In [ ]:
# Step 2: Create house age categories
df["AgeGroup"] = pd.cut(df["HouseAge"], bins=[0, 15, 30, 45, 60],
                         labels=["0-15", "16-30", "31-45", "46-60"])

df[["HouseAge", "AgeGroup"]].head()

In [ ]:
# Step 3: Pivot table - Average house value by Income Group
pivot1 = pd.pivot_table(df, values="MedianHouseValue", index="IncomeGroup", aggfunc="mean")
pivot1

In [ ]:
# Step 4: Pivot table - Average house value by Income Group and Age Group
pivot2 = pd.pivot_table(df, values="MedianHouseValue", index="AgeGroup",
                         columns="IncomeGroup", aggfunc="mean")
pivot2

In [ ]:
# Step 5: Visualize the pivot table as a heatmap
plt.figure(figsize=(8, 5))
sns.heatmap(pivot2, annot=True, fmt=".0f", cmap="YlOrRd")
plt.title("Average Median House Value by Age Group and Income Group")
plt.show()

**Interpretation:** The pivot table shows that as income group increases from "Low" to
"Very High", the average house value increases too. This makes sense — richer
neighbourhoods tend to have more expensive houses.

## Lab Summary

In this lab, we used the **real California Housing dataset** to:
1. Understand the problem of studying house prices.
2. Load the dataset using `scikit-learn`.
3. Identify dependent and independent variables.
4. Compare mean and median to detect skew.
5. Compare variance and IQR to detect spread and outliers.
6. Compute skewness to describe the shape of the distribution.
7. Visualize the distribution using histogram, boxplot, and density plot.
8. Build pivot tables to summarize data by groups.

### Practice Exercises
1. Find the mean and median of `AveRooms`. Is it symmetric or skewed?
2. Calculate the IQR of `Population` and count the outliers.
3. Make a histogram and boxplot of `HouseAge`.
4. Create a pivot table showing average `AveRooms` by `IncomeGroup`.
5. Calculate skewness of `AveOccup` and interpret the result.